# 02 - CLV Model: Predicting Future Customer Value

**Two-stage XGBoost, validated on a temporal holdout**

---

### A note on this notebook's history

An earlier version of this notebook trained a single-stage model:

```python
X = df[["recency", "frequency", "cluster"]]
y = df["monetary"]
```

This has two real problems, both explained and fixed below rather than hidden:

1. **Target leakage.** `cluster` was produced by K-Means fitted *on* `monetary`
   in the previous notebook. Using it to predict `monetary` means one of the
   features is derived from the answer.
2. **Not a forecast.** Predicting `monetary` from `recency`/`frequency` measured
   in the *same* time window isn't prediction - those quantities are
   mechanically correlated inside a fixed window (a customer with high frequency
   in a period necessarily tends to have high monetary in that same period), so
   a high R-squared there mostly reflects a tautology, not skill at forecasting.

This version predicts money a customer will spend in the next six months,
using only features built from an *earlier* window than the target - a genuine
temporal holdout, the way customer-value models are meant to be validated.

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier, XGBRegressor

pd.set_option("display.max_columns", None)

OUT = Path("Outputs")
FEATURES = [
    "recency", "frequency", "monetary", "tenure", "avg_order_value",
    "n_items", "n_products", "purchase_rate", "avg_gap",
]
SEED = 42

## 1. Load the training data

Built in `01_data_prep.ipynb`: features come from an 18-month calibration
window (Dec 2009 - Jun 2011), and `future_value` is what each customer
actually spent in the following 6-month holdout window (0 if they never
came back).

Roughly half of customers spend nothing in the holdout - customer value is
zero-inflated. That shapes the modelling approach below: most of the
uncertainty is *whether* a customer returns at all, not how much they spend
once they do.

In [2]:
train = pd.read_csv(OUT / "clv_training_data.csv")
X = train[FEATURES].values
y = train["future_value"].values
y_bin = (y > 0).astype(int)

print(f"Customers: {len(train):,} | returned during holdout: {y_bin.mean():.1%}")
train.describe()

Customers: 4,966 | returned during holdout: 52.0%


,customer_id,recency,frequency,monetary,tenure,avg_order_value,n_items,n_products,purchase_rate,avg_gap,future_value
count,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000,4966.000000
mean,15339.951470,172.115787,5.284333,2482.715235,363.782118,379.686614,1524.240636,70.542690,0.525077,151.233139,931.480687
std,1704.101435,145.018203,10.236759,11043.097077,156.755738,519.405118,7434.372036,98.056147,1.356028,128.071635,5633.170035
min,12346.000000,1.000000,1.000000,2.900000,1.000000,2.900000,1.000000,1.000000,0.054054,1.000000,0.000000
25%,13879.250000,39.000000,1.000000,318.090000,239.000000,182.152083,166.000000,17.000000,0.142180,58.000000,0.000000
50%,15334.500000,149.000000,3.000000,779.455000,400.000000,289.716667,426.000000,40.000000,0.275229,109.000000,93.395000
75%,16816.750000,250.500000,6.000000,2011.955000,499.000000,425.005125,1123.750000,87.750000,0.517241,211.000000,689.982500
max,18287.000000,555.000000,261.000000,424805.980000,555.000000,14844.766667,251910.000000,2004.000000,30.000000,555.000000,184015.670000


## 2. Two-stage model

Rather than one regressor trying to predict a target that is mostly zeros, this
splits the problem into two better-posed questions:

expected value = P(returns) x E[spend | returns]

1. **Classifier** - will this customer return at all? (XGBoost, all customers)
2. **Regressor** - given that they return, how much will they spend?
   (XGBoost, fitted only on customers who *did* return, target log1p-transformed
   since spend is right-skewed)

Both trained on an 80/20 split of the calibration data, held out and untouched
until evaluation.

In [3]:
tr, te = train_test_split(
    np.arange(len(train)), test_size=0.2, random_state=SEED, stratify=y_bin
)

clf = XGBClassifier(random_state=SEED, verbosity=0, eval_metric="logloss")
clf.fit(X[tr], y_bin[tr])
proba = clf.predict_proba(X[te])[:, 1]

returners = tr[y_bin[tr] == 1]
reg = XGBRegressor(random_state=SEED, verbosity=0)
reg.fit(X[returners], np.log1p(y[returners]))

expected = np.expm1(reg.predict(X[te])).clip(0)
predicted_value = proba * expected

print("Stage 1 (classifier) trained on", len(tr), "customers")
print("Stage 2 (regressor) trained on", len(returners), "returning customers only")

Stage 1 (classifier) trained on 3972 customers
Stage 2 (regressor) trained on 2064 returning customers only


## 3. Evaluation

**Why not report dollar-space R-squared?** It was tested and rejected as the
headline metric: across 5-fold cross-validation it swung from -0.47 to 0.91,
because a single customer spending $184,016 dominates whichever fold happens
to contain them. Any R-squared quoted from one split of this data is mostly an
artifact of that split, not a stable measure of model quality.

AUC (does the classifier rank returners above non-returners?) and top-20%
revenue capture (if you contact the top-ranked 20% of customers, what share of
actual future revenue do you reach?) are stable across folds, so those are
reported instead.

Critically, this notebook also compares against the simplest possible
baseline - just sort customers by how much they spent in the calibration
window - because a model that doesn't beat that isn't earning its complexity.

In [4]:
auc = roc_auc_score(y_bin[te], proba)
pr_auc = average_precision_score(y_bin[te], proba)
rho = float(spearmanr(predicted_value, y[te]).statistic)

n20 = int(len(te) * 0.20)
capture = y[te][np.argsort(-predicted_value)[:n20]].sum() / y[te].sum()
baseline = y[te][np.argsort(-train["monetary"].values[te])[:n20]].sum() / y[te].sum()

# A single train/test split isn't enough evidence of stability -- cross-validate.
cv_aucs = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y_bin):
    m = XGBClassifier(random_state=SEED, verbosity=0, eval_metric="logloss")
    m.fit(X[a], y_bin[a])
    cv_aucs.append(roc_auc_score(y_bin[b], m.predict_proba(X[b])[:, 1]))

print("Model evaluation (temporal holdout)")
print(f"  Return AUC           : {auc:.4f}   (5-fold CV {np.mean(cv_aucs):.4f} +/- {np.std(cv_aucs):.4f})")
print(f"  Return PR-AUC        : {pr_auc:.4f}")
print(f"  Spearman vs actual   : {rho:.4f}")
print(f"  Top-20% capture      : {capture:.1%}")
print(f"  Baseline (past spend): {baseline:.1%}   <- model must beat this to be worth using")

Model evaluation (temporal holdout)
  Return AUC           : 0.7927   (5-fold CV 0.7759 +/- 0.0178)
  Return PR-AUC        : 0.8208
  Spearman vs actual   : 0.6071
  Top-20% capture      : 64.5%
  Baseline (past spend): 65.2%   <- model must beat this to be worth using


### The honest finding

For ranking customers by future value, this model does not clearly beat
sorting by past spend - the gap between `capture` and `baseline` is inside the
noise you'd expect from cross-validation variance. That's disclosed here rather
than glossed over.

Where the model does add something a sort cannot: a calibrated probability
that a given customer returns at all, which supports decisions a ranking
can't - retention budget per customer, expected-value thresholds, distinguishing
"high past spend, probably gone" from "high past spend, coming back".

In [5]:
if capture <= baseline:
    print("For ranking customers, sorting by past spend performs about as well as this model.")
    print("The model's distinct contribution is the calibrated return probability (AUC above),")
    print("not superior ranking of who to contact.")
else:
    print(f"Model beats the naive baseline by {capture - baseline:.1%} on top-20% revenue capture.")

For ranking customers, sorting by past spend performs about as well as this model.
The model's distinct contribution is the calibrated return probability (AUC above),
not superior ranking of who to contact.


## 4. Refit on all data, score every customer

With the evaluation numbers now honestly reported above, refit on the full
calibration set (no holdout needed anymore) and score the entire customer base
for use in the dashboard.

In [6]:
clf_full = XGBClassifier(random_state=SEED, verbosity=0, eval_metric="logloss")
clf_full.fit(X, y_bin)

reg_full = XGBRegressor(random_state=SEED, verbosity=0)
reg_full.fit(X[y_bin == 1], np.log1p(y[y_bin == 1]))

score = pd.read_csv(OUT / "clv_scoring_features.csv")
Xs = score[FEATURES].values
score["return_probability"] = clf_full.predict_proba(Xs)[:, 1]
score["expected_spend"] = np.expm1(reg_full.predict(Xs)).clip(0)
score["predicted_value"] = (score["return_probability"] * score["expected_spend"]).round(2)

score[["customer_id", "predicted_value"]].to_csv(OUT / "predicted_customer_value.csv", index=False)
score[["customer_id", "return_probability", "expected_spend", "predicted_value"]].round(4).to_csv(
    OUT / "clv_predictions_detail.csv", index=False
)

print(f"Scored {len(score):,} customers")
score.sort_values("predicted_value", ascending=False).head(10)

Scored 5,878 customers


,customer_id,recency,frequency,monetary,tenure,avg_order_value,n_items,n_products,purchase_rate,avg_gap,return_probability,expected_spend,predicted_value
5692,18102.0,1,145,608821.65,739,4198.770000,188340,382,5.886333,5.096552,0.999606,143366.984375,143310.437500
2277,14646.0,2,151,528602.52,737,3500.678940,367193,961,6.146540,4.880795,0.999606,129438.195312,129387.140625
5109,17511.0,3,60,175603.55,738,2926.725833,119656,657,2.439024,12.300000,0.999416,110634.312500,110569.710938
5050,17450.0,8,51,246973.09,438,4842.609608,84720,144,3.493151,8.588235,0.998521,109163.554688,109002.093750
2538,14911.0,1,398,295972.63,739,743.649824,149987,2550,16.156969,1.856784,0.999348,88173.453125,88115.953125
4295,16684.0,4,55,147142.77,732,2675.323091,104810,184,2.254098,13.309091,0.999118,74867.234375,74801.187500
1789,14156.0,10,156,313946.37,739,2012.476731,165992,1446,6.332882,4.737179,0.999539,67373.187500,67342.140625
1331,13694.0,4,143,196482.81,735,1374.005664,189205,896,5.836735,5.139860,0.999546,61874.398438,61846.281250
1731,14096.0,4,17,65164.79,102,3833.222941,16352,1119,5.000000,6.000000,0.999514,56892.777344,56865.121094
400,12748.0,1,336,56599.39,735,168.450565,39108,2286,13.714286,2.187500,0.994840,56447.441406,56156.199219


## 5. Save metrics and model artifact

In [7]:
metrics = {
    "model": "xgboost_two_stage",
    "validation": "temporal holdout (18mo calibration -> 6mo holdout)",
    "return_auc": round(float(auc), 4),
    "return_auc_cv_mean": round(float(np.mean(cv_aucs)), 4),
    "return_auc_cv_std": round(float(np.std(cv_aucs)), 4),
    "return_pr_auc": round(float(pr_auc), 4),
    "spearman_vs_actual_future_value": round(rho, 4),
    "top20pct_revenue_capture": round(float(capture), 4),
    "baseline_top20pct_revenue_capture": round(float(baseline), 4),
    "n_train": int(len(tr)),
    "n_test": int(len(te)),
    "features": FEATURES,
    "target": "spend in next 6 months",
}
with open(OUT / "model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(OUT / "model_xgb.pkl", "wb") as f:
    pickle.dump({"classifier": clf_full, "regressor": reg_full, "features": FEATURES}, f)

print(json.dumps(metrics, indent=2))

{
  "model": "xgboost_two_stage",
  "validation": "temporal holdout (18mo calibration -> 6mo holdout)",
  "return_auc": 0.7927,
  "return_auc_cv_mean": 0.7759,
  "return_auc_cv_std": 0.0178,
  "return_pr_auc": 0.8208,
  "spearman_vs_actual_future_value": 0.6071,
  "top20pct_revenue_capture": 0.6449,
  "baseline_top20pct_revenue_capture": 0.6516,
  "n_train": 3972,
  "n_test": 994,
  "features": [
    "recency",
    "frequency",
    "monetary",
    "tenure",
    "avg_order_value",
    "n_items",
    "n_products",
    "purchase_rate",
    "avg_gap"
  ],
  "target": "spend in next 6 months"
}


## Summary

| Metric | Value |
|---|---|
| Return AUC (holdout) | see output above |
| Return AUC (5-fold CV) | see output above - check the +/- is small, i.e. stable |
| Top-20% revenue capture | compared directly against a naive past-spend baseline |

Files written: `model_metrics.json`, `model_xgb.pkl`,
`predicted_customer_value.csv`, `clv_predictions_detail.csv` - all consumed by
the dashboard (`src/app.py`).